# 5. Airtime utilization heatmap by date and band

Takes **date** and **frequency band** (e.g. 539MHz) as input. Reads only CSV Data files for that date and band, then builds one heatmap (like the reference):
- **X axis:** frequency (GHz) — channel center frequencies within the selected band
- **Y axis:** time — hour of day (00:00 at bottom to 23:00 at top)
- **Color:** airtime utilization (%), with a color scale (colorbar) on the right mapping color to number.

In [144]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
from datetime import datetime, timezone
import plotly.graph_objects as go

In [145]:
# Inputs: date and frequency band (only data for this band will be used)
date_input = "2026-02-03"   # YYYY-MM-DD or YYYY_MM_DD
band_input = "3765MHz"      # 3765MHz has more AU data for Feb 1; e.g. 2441MHz, 539MHz

# Normalize date to YYYY_MM_DD for filename matching
date_str = date_input.replace("-", "_").strip()
if len(date_str) == 10 and date_str[4] == "_" and date_str[7] == "_":
    pass  # already YYYY_MM_DD
elif len(date_str) == 8 and date_str.isdigit():
    date_str = f"{date_str[:4]}_{date_str[4:6]}_{date_str[6:8]}"
else:
    raise ValueError(f"Invalid date format: {date_input}. Use YYYY-MM-DD or YYYY_MM_DD.")

# Normalize band: ensure form like 539MHz
band_str = str(band_input).strip()
if re.search(r"^\d+$", band_str):
    band_str = f"{band_str}MHz"
elif not re.search(r"MHz$", band_str, re.IGNORECASE):
    band_str = f"{band_str}MHz"

# Path to CSVs (same as other notebooks)
csvs_dir = Path("work_dir/csvs")
if not csvs_dir.exists():
    csvs_dir = Path("../work_dir/csvs")
occupancy_dir = csvs_dir / "channel_occupancy"
legend_dir = csvs_dir / "frequency_legends"

if not occupancy_dir.exists():
    raise FileNotFoundError(f"Directory not found: {occupancy_dir}")

# Data files for this date and band (transform writes date into filename from parquet mtime)
pattern = f"Data_{date_str}_*_{band_str}_*.csv"
data_files = sorted(occupancy_dir.glob(pattern))

if not data_files:
    print(f"No Data_* files found for date {date_str} and band {band_str} in {occupancy_dir}")
else:
    print(f"Date: {date_input}  |  Band: {band_str}")
    print(f"Found {len(data_files)} Data file(s).")

Date: 2026-02-03  |  Band: 3765MHz
Found 17 Data file(s).


In [146]:
# Get date, hour, band from Data CSV filename (transform writes Data_YYYY_MM_DD_HH_band_dBm.csv from parquet mtime)
def date_hour_band_from_data_path(path: Path):
    m = re.match(r"Data_(\d{4}_\d{2}_\d{2})_(\d{2})_(\d+MHz)_.*\.csv", path.name, re.IGNORECASE)
    if m:
        return m.group(1), int(m.group(2)), m.group(3)  # date_part, hour, band
    return None

# Load each Data file and its Legend: get hour, AU per channel, freq centers (GHz)
# Legend: columns = channel_index, start_Hz, end_Hz -> center = (start+end)/2 / 1e9
hour_au = {}   # hour -> array of AU per channel
freq_centers_ghz = None  # set from first Legend (same band => same layout)

for path in data_files:
    parsed = date_hour_band_from_data_path(path)
    if parsed is None:
        continue
    date_part, hour, band = parsed
    try:
        df = pd.read_csv(path, header=None)
    except Exception:
        continue
    if len(df) < 1:
        continue
    row0 = df.iloc[0].values
    n_col = len(row0)
    n_chans = n_col - 6
    if n_chans < 1:
        continue
    au = np.asarray(row0[5 : 5 + n_chans])

    # Load Legend for this (date, hour, band) to get frequency centers in GHz
    legend_path = legend_dir / f"Legend_{date_part}_{hour:02d}_{band}.csv"
    if legend_path.exists():
        leg = pd.read_csv(legend_path, header=None)
        # columns: channel_index, start_Hz, end_Hz
        centers_hz = (leg.iloc[:, 1].values + leg.iloc[:, 2].values) / 2
        freq_ghz = centers_hz / 1e9
        if freq_centers_ghz is None:
            freq_centers_ghz = freq_ghz
        if len(au) == len(freq_ghz):
            hour_au[hour] = au
    else:
        # No legend: use channel indices as placeholder; still store AU
        if freq_centers_ghz is None:
            freq_centers_ghz = np.arange(n_chans, dtype=float) / 1e3  # fake GHz
        hour_au[hour] = au

if not hour_au or freq_centers_ghz is None:
    raise ValueError("No valid Data+Legend rows for this date and band.")

# Full band extent (MHz) for X-axis: graph spans entire band (e.g. 3765MHz -> 3550–3980)
BAND_EXTENT_MHZ = {
    "195MHz": (180, 220),
    "539MHz": (470, 610),
    "915MHz": (902, 928),
    "2441MHz": (2400, 2484),
    "3765MHz": (3550, 3980),
    "5500MHz": (5150, 5860),
}
f_min_mhz, f_max_mhz = BAND_EXTENT_MHZ.get(band_str, (int(freq_centers_ghz.min() * 1000 - 50), int(freq_centers_ghz.max() * 1000 + 50)))
channel_width_mhz = 20
# Full-band channel centers (GHz), 20 MHz step
full_freq_centers_ghz = np.arange((f_min_mhz + channel_width_mhz // 2) / 1000, f_max_mhz / 1000, channel_width_mhz / 1000)
n_full = len(full_freq_centers_ghz)

# Build matrix over full band: map Legend AU onto full-band grid; channels with no data = 0
hours_sorted = sorted(hour_au.keys())
matrix = np.zeros((len(hours_sorted), n_full))
match_tol_ghz = 0.015  # same 20 MHz channel
for i, hour in enumerate(hours_sorted):
    au_row = hour_au[hour]
    for j, fc in enumerate(full_freq_centers_ghz):
        idx = np.argmin(np.abs(freq_centers_ghz - fc))
        if np.abs(freq_centers_ghz[idx] - fc) < match_tol_ghz and idx < len(au_row):
            matrix[i, j] = au_row[idx]
        # else: 0 (no data above power threshold for this frequency)

print(f"Hours: {hours_sorted}")
print(f"Full band: {f_min_mhz}–{f_max_mhz} MHz ({band_str})")
print(f"Frequency (GHz): {full_freq_centers_ghz[0]:.3f} ... {full_freq_centers_ghz[-1]:.3f} ({n_full} channels)")
print(f"Matrix shape: {matrix.shape} (hours x full-band channels)")

Hours: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 19, 20, 21, 22, 23]
Full band: 3550–3980 MHz (3765MHz)
Frequency (GHz): 3.560 ... 3.960 (21 channels)
Matrix shape: (17, 21) (hours x full-band channels)


In [147]:
# Heatmap with Plotly: X = full band Freq (GHz), Y = Time (hour), color = AU (%). 0% = no data above threshold.
valid = matrix[matrix > 0]
vmax = max(10.0, np.max(matrix) * 1.2) if valid.size else 100.0
hour_labels = [f"{h:02d}:00" for h in hours_sorted]

fig = go.Figure(data=go.Heatmap(
    x=full_freq_centers_ghz,
    y=hour_labels,
    z=matrix,
    colorscale="Viridis",
    zmin=0,
    zmax=vmax,
    colorbar=dict(title="Airtime utilization (%)"),
    hovertemplate="Freq: %{x:.3f} GHz<br>Time: %{y}<br>AU: %{z:.2f}%<extra></extra>",
))
fig.update_layout(
    title=f"Airtime utilization (%) — {date_input} — {band_str}",
    xaxis_title="Freq (GHz)",
    yaxis_title="Time",
    height=600,
    yaxis=dict(autorange="reversed"),
)
fig.show()

### Verification: power threshold and “no data” on the graph

- **Transform behavior:** The transform only writes a frequency channel to the Legend/Data CSVs when there was **at least one raw detection above the power threshold** (e.g. −85 dBm) in that 20 MHz channel for that hour. Channels with **no** raw data above threshold are **not** in the Legend/Data.
- **On this full-band heatmap:** The X-axis spans the **entire band** (e.g. 3550–3980 MHz for 3765 MHz). For each 20 MHz slot:
  - If the transform had a channel there (data above threshold), we show the AU % from the Data CSV.
  - If there was no such channel (no data above threshold), we show **0% AU**.
- So **yes**: frequencies that had no raw data above the power threshold are represented on the graph as **0%** airtime utilization; they are not omitted.